In [ ]:
%%bash

pip install -r ../requirements.txt

In [ ]:
# Connecting To Weaviate
import weaviate
from weaviate.classes.init import Auth
import os

weaviate_api_key=os.getenv("WEAVIATE_API_KEY")
weaviate_cluster=os.getenv("WEAVIATE_CLUSTER")

WEAVIATE_URL = weaviate_cluster
WEAVIATE_API_KEY = weaviate_api_key

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY)
)

print(f"connected to weaviate: {client.is_ready()}")

In [ ]:
# fixing unicode error in google colab
import locale

locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
# specify embedding model (using huggingface sentence transformer)
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model_name = "sentence-transformers/all-mpnet-base-v2"

embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

In [ ]:
# extract data and load
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/data/rag_research_paper.pdf", extract_images=True)
pages = loader.load()
print(pages)

In [ ]:
# Split the text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=20)

docs = text_splitter.split_documents(pages)
print(docs)

In [ ]:
# metadata cleaning
import re

def clean_metadata(metadata: dict) -> dict:
    cleaned = {}
    for key, value in metadata.items():
        # Replace dots and invalid chars with underscore
        new_key = re.sub(r'[^_A-Za-z0-9]', '_', key)
        # If starts with digit, prefix with underscore
        if new_key and new_key[0].isdigit():
            new_key = '_' + new_key
        if new_key:
            cleaned[new_key] = value
    return cleaned

# Clean BEFORE inserting
for doc in docs:
    doc.metadata = clean_metadata(doc.metadata)

In [ ]:
# embedding and storing in vector db
from langchain_weaviate import WeaviateVectorStore

vector_db = WeaviateVectorStore.from_documents(docs, embeddings, client=client, by_text=False)

In [ ]:
print(
    vector_db.similarity_search(
        "What is RAG?",
        k=3
    )[0].page_content
)

In [ ]:
print(
    vector_db.similarity_search(
        "What is RAG?",
        k=3
    )[1].page_content
)

In [ ]:
print(
    vector_db.similarity_search(
        "What is RAG?",
        k=3
    )[2].page_content
)

In [ ]:
print(
    vector_db.similarity_search(
        "What is attention?",
        k=3
    )
)

In [ ]:
# Prompt template
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

prompt = ChatPromptTemplate.from_template(template)
print(prompt)

In [ ]:
# loading the mistral model
from google.colab import userdata
from langchain_huggingface import HuggingFaceEndpoint

HUGGING_FACE_API_TOKEN = userdata.get('HUGGING_FACE_API_TOKEN')

model = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.1",
    huggingfacehub_api_token=HUGGING_FACE_API_TOKEN,
    max_new_tokens=180,
    temperature=0.7
)

In [ ]:
# setting up the rag chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

output_parser = StrOutputParser()
retriver = vector_db.as_retriever()

rag_chain = (
    {"context": retriver, "question": RunnablePassthrough()}
    | prompt
    | model
    | output_parser
)

In [ ]:
print(rag_chain.invoke("What is rag application?"))

In [ ]:
print(rag_chain.invoke("How does the RAG model differ from traditional language generation models?"))